# 勇者傳說 — World Map Generator (T4/L4 GPU, Self-Contained)

Generate world map background and node icons for the overworld.

| Asset | Model | Size | Count |
|-------|-------|------|-------|
| World Map BG | SDXL + LoRA (0.1) | 1024×768 | 1 |
| Node Icons | SDXL + LoRA (0.7) + RMBG | 256→80×80 | 11 |

**Self-contained**: No repo clone needed.  
**Output**: Google Drive `/MyDrive/ai-rpg-game/outputs/worldmap/`  
**Resume**: 中斷後重跑自動跳過已完成的

In [ ]:
# ── Install packages ──
# After this cell runs, the runtime will auto-restart.
# When it restarts, skip this cell and run from the next one.
!pip install -q "numpy<2" scipy  # MUST be first — fixes numpy/scipy C-extension mismatch
!pip install -q diffusers transformers accelerate safetensors "Pillow>=10,<12" matplotlib rembg onnxruntime

# Auto-restart runtime so the new numpy is loaded into memory
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 102.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but 

In [1]:
# ── Imports & Setup (run this after runtime restart) ──
import gc, json, os, time
from pathlib import Path
import numpy as np
from PIL import Image

print(f'numpy version: {np.__version__}')  # should be 1.x

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

GDRIVE_BASE = Path('/content/drive/MyDrive/ai-rpg-game')
OUTPUT_BASE = GDRIVE_BASE / 'outputs' if ON_COLAB else Path('outputs')
MODELS_DIR = GDRIVE_BASE / 'models' if ON_COLAB else Path('models')

for d in [OUTPUT_BASE / 'worldmap', MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    DEVICE = 'cuda'
    DTYPE = torch.float16
    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
    print(f'GPU: {name} ({vram:.1f} GB VRAM)')
else:
    DEVICE = 'cpu'
    DTYPE = torch.bfloat16
    print('WARNING: No GPU!')

print(f'Output: {OUTPUT_BASE / "worldmap"}')

numpy version: 2.0.2
Mounted at /content/drive
GPU: Tesla T4 (14.6 GB VRAM)
Output: /content/drive/MyDrive/ai-rpg-game/outputs/worldmap


In [2]:
class ProgressTracker:
    def __init__(self, task_name, output_dir):
        self.file = Path(output_dir) / f'_progress_{task_name}.json'
        self.completed = set()
        self.start_time = time.time()
        if self.file.exists():
            try:
                data = json.loads(self.file.read_text())
                self.completed = set(data.get('completed', []))
                print(f'[resume] {len(self.completed)} items already done')
            except Exception:
                pass

    def is_done(self, name): return name in self.completed

    def mark_done(self, name):
        self.completed.add(name)
        self.file.parent.mkdir(parents=True, exist_ok=True)
        self.file.write_text(json.dumps({
            'completed': sorted(self.completed),
            'count': len(self.completed),
            'last_updated': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, indent=2))

    def summary(self, total):
        done = len(self.completed)
        elapsed = time.time() - self.start_time
        if done > 0:
            remaining = (total - done) * (elapsed / done)
            return f'{done}/{total} done, ~{remaining/60:.0f} min remaining'
        return f'0/{total} done'


def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def print_vram():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated(0) / 1024**3
        props = torch.cuda.get_device_properties(0)
        total = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
        print(f'[vram] {used:.1f}/{total:.1f} GB')

## Prompt Data (Embedded)

In [3]:
PROMPTS = {
    "_meta": {
        "negative_prompt_xl": "blurry, photorealistic, text, watermark, modern, sci-fi, low quality, jpeg artifacts"
    },
    "worldmap_bg": {
        "name": "worldmap_bg",
        "prompt_xl": "aged medieval fantasy map, weathered parchment paper, ornate golden border frame with corner ornaments, compass rose, ocean areas at edges with sea monsters, coffee stains, fold creases, cartography style, antique map illustration, ink drawings, warm sepia tones",
        "width": 1024, "height": 768,
        "lora_weight": 0.1,
        "needs_rmbg": False,
        "steps_xl": 30,
        "guidance_xl": 7.5,
    },
    "node_icons": [
        {"name": "node_castle", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, medieval stone castle fortress with towers and flags, fantasy game icon, top-down map view"},
        {"name": "node_forest", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, enchanted magical forest with glowing ancient trees, fantasy game icon, top-down map view"},
        {"name": "node_mountain", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, rugged mountain peak with rocky cliffs, fantasy game icon, top-down map view"},
        {"name": "node_volcano", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, erupting volcano with lava and smoke, fantasy game icon, top-down map view"},
        {"name": "node_water", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, ocean cave entrance with crashing waves and coral, fantasy game icon, top-down map view"},
        {"name": "node_skull", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, dark undead territory with skulls and dead trees, fantasy game icon, top-down map view"},
        {"name": "node_demon", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, dark demon castle with evil spires and red glow, fantasy game icon, top-down map view"},
        {"name": "node_treant", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, ancient treant guardian with mossy bark and glowing eyes, fantasy game icon, top-down map view"},
        {"name": "node_peak", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, snowy mountain peaks with ice and frost, fantasy game icon, top-down map view"},
        {"name": "node_hotspring", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, steaming natural hot spring with rocks and mist, fantasy game icon, top-down map view"},
        {"name": "node_dwarf", "prompt_xl": "pixel art icon, RPG map marker, centered, single object, dwarven underground fortress entrance carved in stone, fantasy game icon, top-down map view"},
    ]
}

print(f'worldmap_bg: 1 entry')
print(f'node_icons: {len(PROMPTS["node_icons"])} entries')

worldmap_bg: 1 entry
node_icons: 11 entries


In [4]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

SDXL_HUB = 'stabilityai/stable-diffusion-xl-base-1.0'
LORA_HUB = 'nerijs/pixel-art-xl'

# Check Drive cache
sdxl_cache = MODELS_DIR / 'stable-diffusion-xl-base-1.0'
model_path = str(sdxl_cache) if (sdxl_cache / 'model_index.json').exists() else SDXL_HUB

print(f'[pipeline] Loading SDXL ({DEVICE}, {DTYPE})...')
t0 = time.time()

kwargs = {'torch_dtype': DTYPE, 'use_safetensors': True, 'low_cpu_mem_usage': False}
if model_path == SDXL_HUB:
    kwargs['variant'] = 'fp16'

pipe = StableDiffusionXLPipeline.from_pretrained(model_path, **kwargs)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    algorithm_type='dpmsolver++',
    use_karras_sigmas=True,
)

try:
    pipe.load_lora_weights(LORA_HUB, adapter_name='pixel_xl')
    pipe.set_adapters(['pixel_xl'], adapter_weights=[0.8])
    print('[pipeline] pixel-art-xl LoRA loaded (weight=0.8)')
except Exception as e:
    print(f'[warn] LoRA failed: {e}')

pipe = pipe.to(DEVICE)
pipe.enable_attention_slicing()

print(f'[pipeline] Ready in {time.time()-t0:.1f}s')
print_vram()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[pipeline] Loading SDXL (cuda, torch.float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

pixel-art-xl.safetensors:   0%|          | 0.00/171M [00:00<?, ?B/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


[pipeline] pixel-art-xl LoRA loaded (weight=0.8)
[pipeline] Ready in 103.8s
[vram] 6.7/14.6 GB


In [5]:
from PIL import Image

_rembg_session = None

def remove_background(image):
    global _rembg_session
    try:
        from rembg import remove, new_session
        if _rembg_session is None:
            _rembg_session = new_session('u2net')
        return remove(image, session=_rembg_session)
    except Exception:
        print('[warn] No background removal available')
        return image.convert('RGBA')

## Generate World Map Background

Parchment-style map at 1024×768, LoRA weight 0.1 for painterly look.

In [6]:
# ── Generate worldmap_bg (1024x768, LoRA 0.1) ──
entry = PROMPTS['worldmap_bg']
neg = PROMPTS['_meta']['negative_prompt_xl']
out_dir = OUTPUT_BASE / 'worldmap'
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f'{entry["name"]}.png'

if out_path.exists():
    print(f'[skip] {entry["name"]} already exists')
    worldmap_bg = Image.open(out_path)
else:
    # Set LoRA to 0.1 for painterly parchment look
    try:
        pipe.set_adapters(['pixel_xl'], adapter_weights=[entry['lora_weight']])
        print(f'[pipeline] LoRA weight set to {entry["lora_weight"]} for background')
    except Exception:
        pass

    print(f'[gen] {entry["name"]} ({entry["width"]}x{entry["height"]}, {entry["steps_xl"]} steps)')
    t0 = time.time()

    result = pipe(
        prompt=entry['prompt_xl'],
        negative_prompt=neg,
        width=entry['width'],
        height=entry['height'],
        num_inference_steps=entry['steps_xl'],
        guidance_scale=entry['guidance_xl'],
    )
    worldmap_bg = result.images[0]
    worldmap_bg.save(str(out_path), 'PNG')

    kb = out_path.stat().st_size / 1024
    print(f'        saved: {entry["name"]}.png ({kb:.0f} KB, {time.time()-t0:.1f}s)')
    free_vram()

# Restore LoRA weight
try:
    pipe.set_adapters(['pixel_xl'], adapter_weights=[0.8])
except Exception:
    pass

print('Done!')

[pipeline] LoRA weight set to 0.1 for background
[gen] worldmap_bg (1024x768, 30 steps)


  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


        saved: worldmap_bg.png (1457 KB, 31.3s)
Done!


In [ ]:
# ── Generate node icons (11 icons at 256x256, RMBG, downscale to 80x80) ──
entries = PROMPTS['node_icons']
neg = PROMPTS['_meta']['negative_prompt_xl']
out_dir = OUTPUT_BASE / 'worldmap'
out_dir.mkdir(parents=True, exist_ok=True)

tracker = ProgressTracker('worldmap_icons', OUTPUT_BASE)
remaining = sum(1 for e in entries if not tracker.is_done(e['name']))
print(f'Node Icons: {len(entries)} total, {remaining} to generate')

# Set LoRA to 0.7 for pixel art icons
try:
    pipe.set_adapters(['pixel_xl'], adapter_weights=[0.7])
    print('[pipeline] LoRA weight set to 0.7 for icons')
except Exception:
    pass

for i, entry in enumerate(entries):
    name = entry['name']
    out_path = out_dir / f'{name}.png'

    if tracker.is_done(name) and out_path.exists():
        print(f'  [{i+1}/{len(entries)}] [skip] {name}')
        continue

    print(f'\n  [{i+1}/{len(entries)}] {name} -- {tracker.summary(len(entries))}')
    t0 = time.time()

    result = pipe(
        prompt=entry['prompt_xl'],
        negative_prompt=neg,
        width=256, height=256,
        num_inference_steps=30, guidance_scale=7.5,
    )
    image = result.images[0]

    # Remove background
    image = remove_background(image)

    # Trim to bounding box
    if image.mode == 'RGBA':
        bbox = image.getbbox()
        if bbox:
            image = image.crop(bbox)

    # Center on 80x80 canvas using NEAREST
    target = 80
    w, h = image.size
    scale = min(target / w, target / h)
    nw, nh = int(w * scale), int(h * scale)
    image = image.resize((nw, nh), Image.NEAREST)
    canvas = Image.new('RGBA', (target, target), (0, 0, 0, 0))
    canvas.paste(image, ((target - nw) // 2, (target - nh) // 2),
                 image if image.mode == 'RGBA' else None)

    canvas.save(str(out_path), 'PNG')
    kb = out_path.stat().st_size / 1024
    print(f'        saved: {name}.png ({kb:.0f} KB, {time.time()-t0:.1f}s)')
    tracker.mark_done(name)
    free_vram()

# Restore LoRA weight
try:
    pipe.set_adapters(['pixel_xl'], adapter_weights=[0.8])
except Exception:
    pass

print('\nDone!')

Node Icons: 11 total, 11 to generate
[pipeline] LoRA weight set to 0.7 for icons

  [1/11] node_castle -- 0/11 done


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|                                               | 0.00/176M [00:00<?, ?B/s]

        saved: node_castle.png (3 KB, 78.8s)

  [2/11] node_forest -- 1/11 done, ~13 min remaining


  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

        saved: node_peak.png (4 KB, 12.7s)

  [10/11] node_hotspring -- 9/11 done, ~1 min remaining


  0%|          | 0/30 [00:00<?, ?it/s]

        saved: node_hotspring.png (4 KB, 12.7s)

  [11/11] node_dwarf -- 10/11 done, ~0 min remaining


  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
# ── Preview: worldmap_bg with icons overlaid ──
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

out_dir = OUTPUT_BASE / 'worldmap'
bg_path = out_dir / 'worldmap_bg.png'

if not bg_path.exists():
    print('No worldmap_bg found yet')
else:
    bg = Image.open(bg_path).convert('RGBA')
    composite = bg.copy()

    # Approximate icon positions (spread across the map)
    icon_positions = [
        ('node_castle',    512, 384),   # center
        ('node_forest',    200, 200),   # top-left area
        ('node_mountain',  780, 180),   # top-right area
        ('node_volcano',   850, 500),   # right area
        ('node_water',     150, 550),   # bottom-left
        ('node_skull',     650, 600),   # bottom-right
        ('node_demon',     900, 350),   # far right
        ('node_treant',    350, 150),   # top area
        ('node_peak',      600, 120),   # top-center
        ('node_hotspring', 300, 500),   # left area
        ('node_dwarf',     500, 600),   # bottom-center
    ]

    labels = []
    for name, x, y in icon_positions:
        icon_path = out_dir / f'{name}.png'
        if icon_path.exists():
            icon = Image.open(icon_path).convert('RGBA')
            # Center icon on position
            ix = x - icon.width // 2
            iy = y - icon.height // 2
            composite.paste(icon, (ix, iy), icon)
            labels.append((name, x, y))

    fig, ax = plt.subplots(1, 1, figsize=(14, 10.5))
    ax.imshow(composite)
    for name, x, y in labels:
        ax.annotate(name.replace('node_', ''), (x, y + 45),
                    ha='center', fontsize=7, color='white',
                    bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6))
    ax.set_title('World Map Preview (approximate positions)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'Icons overlaid: {len(labels)}/{len(icon_positions)}')

## Export: Manifest + Download

In [ ]:
import shutil

out_dir = OUTPUT_BASE / 'worldmap'

# Generate manifest
keys = sorted(f.stem for f in out_dir.glob('*.png')
              if not f.stem.startswith('_'))
manifest = {'worldmap_elements': keys}

manifest_path = out_dir / 'manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Manifest:')
print(f'  worldmap_elements: {len(keys)}')
for k in keys:
    print(f'    - {k}')

# Zip + download
zip_path = '/content/worldmap_output'
shutil.make_archive(zip_path, 'zip', str(out_dir))
print(f'\nZip: {os.path.getsize(zip_path + ".zip") / 1024 / 1024:.1f} MB')

if ON_COLAB:
    from google.colab import files
    files.download(f'{zip_path}.zip')
else:
    print(f'Download: {zip_path}.zip')

Manifest:
  worldmap_elements: 12
    - node_castle
    - node_demon
    - node_dwarf
    - node_forest
    - node_hotspring
    - node_mountain
    - node_peak
    - node_skull
    - node_treant
    - node_volcano
    - node_water
    - worldmap_bg

Zip: 1.5 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>